# Desafio Lighthouse 2026.2 — LH Nautical

Notebook único cobrindo as 7 questões do desafio, seguindo a mesma organização usada na resolução do 2026.1: uma seção markdown por questão, com o raciocínio explicado antes e depois de cada bloco de código — não só o resultado.

**Organização do código (fase de refinamento):** enquanto as respostas ainda estão sendo ajustadas, o código de cada questão mora só em `../Submissão/Q1/`, `../Submissão/Q2/` etc. — os mesmos arquivos que vão pro formulário. Este notebook carrega e executa esses arquivos, narra o raciocínio em markdown. Evita manter duas cópias divergentes (`src/` e `Submissão/`) enquanto a resposta ainda pode mudar; ao final do desafio, os arquivos definitivos de `Submissão/` são copiados para `Workspace/src/` como código de referência do repositório.

**Engine:** DuckDB lendo os CSVs brutos de `../data/raw/1-lh_nautical_csv/` diretamente. A modelagem final em PostgreSQL (`schema.sql`) é construída nas Questões 2 e 3; até lá, o DuckDB serve só para exploração ad hoc sobre o CSV cru, sem inventar tipos definitivos.

**Decisão de robustez importante (detalhada em `../../Anotações/comentarios.md`):** todo `read_csv_auto` deste notebook usa `sample_size=-1` (varredura completa do arquivo para inferir tipos), não o padrão de 20.480 linhas do DuckDB. Confirmei com um teste reproduzível que a amostra padrão pode inferir um tipo errado e quebrar em runtime quando um valor fora do padrão da coluna cai fora da janela amostrada — e `orders.csv` (48.998 linhas) já ultrapassa esse padrão. Como os CSVs deste desafio são pequenos o suficiente (o maior tem 147 mil linhas), a varredura completa custa segundos — não há razão para arriscar.

## Q1 — EDA

### Setup — carregar `orders` no DuckDB

Premissas obrigatórias da Q1: usar apenas a tabela `orders`, sem limpeza/tratamento. `read_csv_auto` só faz inferência de tipo para viabilizar a leitura (necessário para ter uma tabela SQL de verdade) — não corrige, filtra ou descarta nada. `sample_size=-1` garante que essa inferência olhe as 48.998 linhas, não uma amostra.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("""
    CREATE OR REPLACE VIEW orders AS
    SELECT * FROM read_csv_auto('../data/raw/1-lh_nautical_csv/orders.csv', sample_size=-1)
""")

con.sql("DESCRIBE orders")

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ order_number    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ channel         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ salesperson_id  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ location_id     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ status          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ subtotal        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ discount_amount │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ total           │ DOUBLE      │ YES 

### Q1.1 — SQL (Parte 1: visão geral + Parte 2: valores numéricos)

Query em `../Submissão/Q1/1.1.sql` — o mesmo arquivo que vai pro upload da 1.1. Cobre os 5 itens pedidos (linhas, datas min/max, total min/max/médio) mais a contagem de colunas via `information_schema.columns` — pedida na Parte 1 do enunciado principal. Contar colunas assim, e não com `COUNT(*)`, é uma lição direta da revisão do 2026.1: `COUNT(*)` conta linhas, nunca colunas.

In [2]:
with open("../Submissão/Q1/1.1.sql") as f:
    query = f.read()

con.sql(query)

┌──────────────┬───────────────┬─────────────────────┬─────────────────────┬───────────┬───────────┬────────────────────┐
│ total_linhas │ total_colunas │      data_min       │      data_max       │ total_min │ total_max │    total_media     │
│    int64     │     int64     │      timestamp      │      timestamp      │  double   │  double   │       double       │
├──────────────┼───────────────┼─────────────────────┼─────────────────────┼───────────┼───────────┼────────────────────┤
│        48998 │            13 │ 2020-01-01 01:19:28 │ 2026-12-31 23:43:09 │     32.62 │ 127262.02 │ 28704.992077227675 │
└──────────────┴───────────────┴─────────────────────┴─────────────────────┴───────────┴───────────┴────────────────────┘

**Resultado (Parte 1 + Parte 2):**

- Quantidade total de linhas: **48.998**
- Quantidade total de colunas: **13**
- Intervalo de datas (`created_at`): **2020-01-01 01:19:28** a **2026-12-31 23:43:09**
- `total`: mínimo **R$ 32,62** · máximo **R$ 127.262,02** · médio **R$ 28.704,99**

### Q1.2 — Validação

**Qual é o valor médio registrado na coluna "total"?**

R$ 28.704,99

### Investigação de qualidade de dados (apoio à Q1.3)

Não faz parte do código pedido em 1.1 — sustenta o diagnóstico da Parte 3. Cada célula abaixo carrega uma pergunta isolada de `../Submissão/Q1/dq_*.sql`: nulos por coluna e consistência aritmética, se o nulo em `salesperson_id` é estrutural ou falta de dado, distribuição de `status`, outliers em `total` via IQR, e cobertura de datas por ano.

In [3]:
with open("../Submissão/Q1/dq_nulls_and_consistency.sql") as f:
    query = f.read()

con.sql(query)

┌────────────┬──────────────────┬─────────────────────┬──────────────────┬──────────────┬─────────────┬────────────────────┬──────────────────────────────────┬────────────────┬─────────────────────────┐
│ null_total │ null_customer_id │ null_salesperson_id │ null_location_id │ null_channel │ null_status │ total_nao_positivo │ total_inconsistente_com_subtotal │ ids_duplicados │ order_number_duplicados │
│   int64    │      int64       │        int64        │      int64       │    int64     │    int64    │       int64        │              int64               │     int64      │          int64          │
├────────────┼──────────────────┼─────────────────────┼──────────────────┼──────────────┼─────────────┼────────────────────┼──────────────────────────────────┼────────────────┼─────────────────────────┤
│          0 │                0 │               24131 │                0 │            0 │           0 │                  0 │                                0 │              0 │            

In [4]:
with open("../Submissão/Q1/dq_salesperson_null_by_channel.sql") as f:
    query = f.read()

con.sql(query)

┌───────────┬─────────┬──────────────┐
│  channel  │ pedidos │ sem_vendedor │
│  varchar  │  int64  │    int64     │
├───────────┼─────────┼──────────────┤
│ ecommerce │   34342 │        24131 │
│ pos       │   14656 │            0 │
└───────────┴─────────┴──────────────┘

In [5]:
with open("../Submissão/Q1/dq_status_distribution.sql") as f:
    query = f.read()

con.sql(query)

┌───────────┬───────┐
│  status   │   n   │
│  varchar  │ int64 │
├───────────┼───────┤
│ paid      │ 34365 │
│ confirmed │  7335 │
│ cancelled │  4847 │
│ draft     │  2451 │
└───────────┴───────┘

In [6]:
with open("../Submissão/Q1/dq_outliers_iqr.sql") as f:
    query = f.read()

con.sql(query)

┌───────────┬────────────┬────────────────┬─────────────────┐
│    q1     │     q3     │ outliers_acima │ outliers_abaixo │
│  double   │   double   │     int64      │      int64      │
├───────────┼────────────┼────────────────┼─────────────────┤
│ 13171.235 │ 40941.8825 │            452 │               0 │
└───────────┴────────────┴────────────────┴─────────────────┘

In [7]:
with open("../Submissão/Q1/dq_orders_by_year.sql") as f:
    query = f.read()

con.sql(query)

┌───────┬───────┐
│  ano  │   n   │
│ int64 │ int64 │
├───────┼───────┤
│  2020 │  4466 │
│  2021 │  5088 │
│  2022 │  5856 │
│  2023 │  6697 │
│  2024 │  7666 │
│  2025 │  8957 │
│  2026 │ 10268 │
└───────┴───────┘

### Q1.3 — Interpretação (diagnóstico de confiabilidade)

Isoladamente, a tabela `orders` está limpa e internamente consistente: não há valores nulos em `id`, `customer_id`, `location_id`, `channel`, `status`, `total` ou nas colunas de data; `total` nunca é nulo, zero ou negativo; `subtotal - discount_amount` bate com `total` nas 48.998 linhas (0 divergências); e não há `id` nem `order_number` duplicados. O único `NULL` relevante é `salesperson_id`, ausente em 24.131 das 34.342 vendas do canal `ecommerce` e presente em 0% das vendas `pos`. **Correção sobre uma leitura anterior deste diagnóstico:** os nulos ocorrerem só no `ecommerce` não significa que todo pedido `ecommerce` careça de vendedor — 10.211 dos 34.342 pedidos `ecommerce` (~30%) têm `salesperson_id` preenchido. Ou seja, há uma correlação forte (100% dos nulos vêm do canal online), mas não uma regra determinística ("pedido online nunca tem vendedor"). Sem uma regra de negócio explícita (ex.: vendedor só é registrado em venda assistida/por telefone dentro do canal ecommerce), trato essa ausência como um padrão a investigar nas próximas questões, não como uma explicação estrutural fechada — e mantenho em aberto se isso é ou não um problema de qualidade.

**Outliers em `total`:** a distribuição vai de R$32,62 a R$127.262,02, com média (R$28.704,99) puxada acima da mediana (R$25.917,84) por uma cauda de valores altos. Pelo critério de IQR (1,5×), 452 pedidos (~0,92%) ficam acima do limite superior (~R$99.517), e nenhum abaixo do limite inferior. Dado o contexto — varejo náutico, motores de popa e embarcações — pedidos de dezenas de milhares de reais são plausíveis, não parecem erro de digitação (sem negativos, sem sinal de vírgula/ponto trocado). Mas isso não é uma confirmação: só com `orders` não dá pra validar se são de fato pedidos legítimos de grande porte — a legitimidade só pode ser confirmada relacionando esses pedidos com `order_items`, produtos e preços unitários, o que exige o schema completo (Q2/Q3). Recomendação: não descartar automaticamente, mas tratar como "plausível, ainda não validado".

**Ponto a registrar, não necessariamente um erro:** `created_at` vai até 2026-12-31. Contando a partir de hoje (2026-08-10, inclusive), são 4.338 pedidos (~9%) com data igual ou posterior a hoje; estritamente depois de hoje, 4.322 — a diferença são 16 pedidos do próprio dia 10/08. O enunciado avisa que a base é fictícia e cobre 2020–2026, então trato isso como dado sintético gerado para o período todo, não como bug de carga; numa base real isso exigiria checar se são pedidos futuros legítimos (pré-venda) ou erro de ingestão.

**Veredito:** como tabela isolada, `orders` está pronta para as agregações simples pedidas aqui (contagens, min/max/média) — não há tratamento prévio necessário para isso. Não está pronta, porém, para sustentar sozinha as perguntas de negócio das próximas questões: (a) ainda não é possível confirmar que as demais 23 tabelas têm a mesma consistência — em particular `order_items`, cuja soma por pedido deveria reconciliar com `orders.total`, só será verificável depois que Q2/Q3 carregarem o schema completo; (b) pedidos com status `cancelled` (4.847, ~10%) e `draft` (2.451, ~5%) provavelmente não deveriam entrar em métricas de faturamento realizado — esse filtro não foi pedido aqui (a instrução foi "não trate os dados"), mas será necessário nas questões de análise de vendas/clientes à frente.

## Q2 — Schema

**Objetivo:** ler os 24 CSVs brutos e gerar um `schema.sql` (DDL PostgreSQL), um `CREATE TABLE` por arquivo. Premissa obrigatória: só Python 3 + biblioteca padrão (`csv`, `re`, `datetime`, `decimal`, `argparse`, `pathlib` — nada de pandas/dask/polars).

**Regra de inferência — híbrida, não só por valor.** Por coluna: (1) um override semântico explícito por `tabela.coluna` tem prioridade, para identificadores/códigos/documentos que podem *parecer* numéricos mas não são grandezas matemáticas (CPF, CNPJ, telefone, CEP, SKU, EAN, NCM, chave/série de NF-e, números de pedido/compra/devolução); (2) na ausência de override, varredura completa (100% das linhas, nunca amostra) decide o tipo por alargamento incremental: vazio → booleano → inteiro → decimal → data → timestamp → texto. Duas guardas valem tanto pro caminho inteiro quanto pro decimal: zero à esquerda (nenhum tipo numérico do Postgres preserva isso) e limite de `BIGINT` (~9,2×10¹⁸). Raciocínio completo, com os bugs encontrados no caminho, em `../../Anotações/comentarios.md`.

Código em `../Submissão/Q2/infer_schema.py` (entregável da Q2.1); DDL gerado em `../Submissão/Q2/schema.sql` (entregável da Q2.2).

**Resultado:** 24 CSVs processados, 24 `CREATE TABLE` gerados, validado contra Postgres real (não só sintaxe): banco criado do zero, `schema.sql` executado sem erro, os 24 CSVs carregados com `\copy` e reconciliados linha a linha contra a fonte.

| | |
|---|---:|
| Arquivos CSV processados | 24 |
| Tabelas geradas | 24 |
| Total de linhas (CSV, sem cabeçalho) | 433.424 |
| Total de linhas carregadas no Postgres | 433.424 |
| Divergência | 0 em todas as 24 tabelas |

Casos que testaram a robustez da inferência (detalhe completo em `comentarios.md`): `fiscal_invoices.nfe_access_key` (44 dígitos) não estoura `BIGINT` nem vira `NUMERIC` por engano — vai pra `TEXT` por override semântico; `customers.tax_id` tem zero à esquerda em 223 das 2.000 linhas, pego pela guarda de integridade; `employees.cpf` não tem zero à esquerda nos 15 registros atuais (só sorte de amostra) — por isso tem override semântico explícito, não dependendo do dado observado; `fiscal_invoices.series` (sempre `"001"`) e `locations.number`/`addresses.number` (podem ter `S/N` no futuro) também têm override explícito, não só o que a amostra atual mostra.

Uma revisão externa (`Enunciado/Q2/correção.md`) encontrou uma divergência de contagem (eu tinha documentado 311.633 em vez de 433.424 — erro de redação, não de carga, confirmado depois com a reconciliação acima) e dois overrides semânticos faltando (`locations.number`, `fiscal_invoices.series`), já corrigidos no script e regenerados no `schema.sql`.

## Q3 — Carregamento

**Objetivo:** script Python que carrega os 24 CSVs no schema da Q2, sem nenhum tratamento (sem remover nulo, sem corrigir caractere especial). Diferente da Q2, aqui qualquer biblioteca é permitida — usei `psycopg2` com `COPY FROM STDIN`.

**Por que `COPY FROM STDIN`, não `INSERT` nem `COPY FROM` no servidor:** o arquivo é transmitido do cliente Python direto pro protocolo `COPY`, sem reconstruir linha nenhuma em Python — preserva aspas, vírgula interna, acento e zero à esquerda exatamente como estão no CSV. `COPY FROM '/caminho.csv'` leria o filesystem do *servidor* Postgres, não o do script; `INSERT` por linha arriscaria reprocessar o valor sem querer. Colunas do `COPY` são declaradas explicitamente na ordem do cabeçalho do CSV (não presume ordem física da tabela), e identificadores usam `psycopg2.sql.Identifier`, nunca f-string.

**Atomicidade e não-duplicação:** as 24 tabelas carregam numa única transação — `COMMIT` só depois de reconciliar as 24 contagens e 8 checagens de fidelidade; qualquer divergência faz `ROLLBACK` de tudo. Como o schema da Q2 não tem `PRIMARY KEY`/`UNIQUE` (camada bruta, de propósito), o script aborta por padrão se o destino já tiver linha — evita duplicação silenciosa numa segunda execução. `--truncate-before-load` existe pra quando isso for intencional, mas nunca é o padrão.

Código em `../Submissão/Q3/load_data.py` (entregável da Q3.1). Raciocínio completo em `../../Anotações/comentarios.md`.

**Resultado — 6 cenários testados de verdade, não só implementados:**

| Cenário | Resultado |
|---|---|
| Caminho feliz | 24 tabelas, 433.424 linhas, 0 divergência, 8/8 checagens de fidelidade OK, `COMMIT` |
| Segunda execução sem flag | Recusa clara, `exit 1`, `orders` continua em 48.998 (sem duplicar) |
| `--truncate-before-load` | Trunca e recarrega com sucesso (opt-in explícito) |
| Falha no meio (valor incompatível em `attributes.csv`) | Erro com arquivo/tabela/linha/coluna, `ROLLBACK` total — nem `addresses`, já carregada antes, ficou com dado |
| Schema divergente (coluna renomeada em `brands.csv`) | Detectado antes de qualquer `COPY` |
| Credencial inválida | Falha clara, sem senha na saída |

**Q3.2 — soma de linhas de `customers + orders + order_items + payments`, consultada diretamente no PostgreSQL após o `COMMIT`:**

**251.864** (2.000 + 48.998 + 147.320 + 53.546)

## Q4 — Análise de Clientes

**Objetivo:** identificar os 10 clientes "fiéis" — maior ticket médio entre quem comprou de 13 ou mais categorias distintas — e descobrir qual categoria concentra a maior quantidade de itens comprados por esse grupo.

**A decisão que importa aqui não é técnica avançada, é granularidade.** `orders` tem uma linha por pedido; `order_items` tem uma linha por item. Se faturamento e frequência fossem calculados depois de um `JOIN` com `order_items` (necessário pra chegar em categoria), um pedido de 3 itens contaria 3 vezes. Por isso: uma CTE calcula faturamento/frequência/ticket só a partir de `orders`; outra, separada, percorre `orders → order_items → product_variants → products` só pra contar `COUNT(DISTINCT category_id)`. As duas se juntam depois, por `customer_id` — nunca antes.

Segui a definição literal do enunciado para faturamento (soma de `total`, sem filtro de status) — `cancelled`/`draft` continuam na soma porque a questão não pediu esse filtro; numa análise real, essa seria uma pergunta pro negócio, não uma decisão para inventar aqui.

SQL completo (duas consultas independentes e comentadas) em `../Submissão/Q4/4.1.sql`; explicação da cadeia de tabelas, do filtro e da restrição ao Top 10 em `../Submissão/Q4/4.2.md`.

In [8]:
import psycopg2

# Credenciais vem das variaveis de ambiente padrao do Postgres (PGHOST,
# PGDATABASE, PGUSER, ...), nunca hardcoded - mesma convencao do load_data.py da Q3.
pg = psycopg2.connect()

top_10 = pd.read_sql("""
    WITH metricas_pedidos AS (
        SELECT
            customer_id,
            SUM(total) AS faturamento_total,
            COUNT(id) AS frequencia,
            SUM(total) / COUNT(id) AS ticket_medio
        FROM orders
        GROUP BY customer_id
    ),
    diversidade_clientes AS (
        SELECT
            o.customer_id,
            COUNT(DISTINCT p.category_id) AS diversidade_categorias
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.id
        JOIN product_variants pv ON pv.id = oi.product_variant_id
        JOIN products p ON p.id = pv.product_id
        GROUP BY o.customer_id
    )
    SELECT
        mp.customer_id,
        ROUND(mp.faturamento_total, 2) AS faturamento_total,
        mp.frequencia,
        ROUND(mp.ticket_medio, 2) AS ticket_medio,
        dc.diversidade_categorias
    FROM metricas_pedidos mp
    JOIN diversidade_clientes dc ON dc.customer_id = mp.customer_id
    WHERE dc.diversidade_categorias >= 13
    ORDER BY mp.ticket_medio DESC, mp.customer_id ASC
    LIMIT 10
""", pg)

top_10

/var/folders/kc/hb_g153j30gf097ms6nw5w_00000gp/T/ipykernel_7655/1834475957.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  top_10 = pd.read_sql("""


,customer_id,faturamento_total,frequencia,ticket_medio,diversidade_categorias
0,22,1087838.44,26,41839.94,14
1,1477,916262.58,22,41648.30,14
2,929,1082775.89,26,41645.23,14
3,1116,655737.20,16,40983.58,14
4,1691,815471.30,20,40773.57,14
5,774,726127.99,18,40340.44,14
6,1470,1040553.09,26,40021.27,14
7,1599,997616.46,25,39904.66,14
8,965,677297.78,17,39841.05,14
9,1722,1146455.22,29,39532.94,14


In [9]:
categoria_top10 = pd.read_sql("""
    WITH metricas_pedidos AS (
        SELECT customer_id, SUM(total) / COUNT(id) AS ticket_medio
        FROM orders
        GROUP BY customer_id
    ),
    diversidade_clientes AS (
        SELECT o.customer_id, COUNT(DISTINCT p.category_id) AS diversidade_categorias
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.id
        JOIN product_variants pv ON pv.id = oi.product_variant_id
        JOIN products p ON p.id = pv.product_id
        GROUP BY o.customer_id
    ),
    top_10 AS (
        SELECT mp.customer_id
        FROM metricas_pedidos mp
        JOIN diversidade_clientes dc ON dc.customer_id = mp.customer_id
        WHERE dc.diversidade_categorias >= 13
        ORDER BY mp.ticket_medio DESC, mp.customer_id ASC
        LIMIT 10
    )
    SELECT
        c.id AS category_id,
        c.name AS categoria,
        SUM(oi.quantity) AS quantidade_total
    FROM top_10 t
    JOIN orders o ON o.customer_id = t.customer_id
    JOIN order_items oi ON oi.order_id = o.id
    JOIN product_variants pv ON pv.id = oi.product_variant_id
    JOIN products p ON p.id = pv.product_id
    JOIN categories c ON c.id = p.category_id
    GROUP BY c.id, c.name
    ORDER BY quantidade_total DESC, category_id ASC
""", pg)

pg.close()
categoria_top10

/var/folders/kc/hb_g153j30gf097ms6nw5w_00000gp/T/ipykernel_7655/2517821006.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  categoria_top10 = pd.read_sql("""


,category_id,categoria,quantidade_total
0,8,Hélices,492.0
1,3,Coletes Salva-Vidas,393.0
2,5,Eletrônica Náutica,392.0
3,7,Âncoras,387.0
4,10,Iluminação,333.0
5,12,Manutenção,330.0
6,13,SEGURANÇA,325.0
7,6,Velas,313.0
8,11,Pintura Marítima,307.0
9,9,Acessórios de Convés,305.0


**Resultado:** categoria líder entre os 10 clientes de elite — **Hélices** (`category_id = 8`), 492 itens somados. A segunda colocada (Coletes Salva-Vidas, 393) fica bem atrás, então não há empate no primeiro lugar.

Validado contra os números de referência do enunciado antes de considerar pronto: 14 categorias distintas em `products`, 1.971 clientes com diversidade ≥ 13, e a cardinalidade dos joins obrigatórios conferida (`order_items` mantém 147.320 linhas nas 3 etapas de join, sem perder nem duplicar) — raciocínio completo em `../../Anotações/comentarios.md`.

## Q5 — Dimensão de Calendário

**Objetivo:** o Sr. Almir quer saber qual dia da semana tem a pior média de vendas nas lojas físicas (`channel = 'pos'`), pra decidir se vale fechar a loja nesse dia. Um estagiário fictício agrupou direto em `orders` e achou Domingo ótimo — mas dias em que a loja abriu e vendeu zero não existem em `orders` (não é linha com valor zero, é ausência de linha), então saem do cálculo e inflam a média.

**A correção é uma dimensão de calendário**, gerada dentro da própria consulta com `generate_series` (uma linha por data entre o `MIN` e o `MAX` de `placed_at::date` observados no arquivo — não `CURRENT_DATE`, pra consulta ficar reproduzível e não ignorar pedidos com data futura já presentes no arquivo). O `LEFT JOIN` parte do calendário, não de `orders`, e `COALESCE(venda_diaria, 0)` transforma a ausência de venda em zero antes da média — sem isso, `AVG()` ignoraria justamente os dias que o calendário existe pra preservar.

SQL completo em `../Submissão/Q5/5.1.sql`; explicação em `../Submissão/Q5/5.2.md`.

In [10]:
pg = psycopg2.connect()

calendario_semana = pd.read_sql("""
    WITH limites AS (
        SELECT MIN(placed_at::date) AS data_inicial, MAX(placed_at::date) AS data_final
        FROM orders
    ),
    datas AS (
        SELECT generate_series(data_inicial, data_final, INTERVAL '1 day')::date AS data
        FROM limites
    ),
    calendario AS (
        SELECT
            data,
            EXTRACT(ISODOW FROM data)::int AS numero_dia_semana,
            CASE EXTRACT(ISODOW FROM data)::int
                WHEN 1 THEN 'Segunda-feira' WHEN 2 THEN 'Terça-feira' WHEN 3 THEN 'Quarta-feira'
                WHEN 4 THEN 'Quinta-feira' WHEN 5 THEN 'Sexta-feira' WHEN 6 THEN 'Sábado'
                WHEN 7 THEN 'Domingo'
            END AS dia_semana
        FROM datas
    ),
    vendas_diarias AS (
        SELECT placed_at::date AS data, SUM(total) AS venda_diaria
        FROM orders
        WHERE channel = 'pos'
        GROUP BY placed_at::date
    ),
    calendario_com_vendas AS (
        SELECT c.data, c.numero_dia_semana, c.dia_semana, COALESCE(v.venda_diaria, 0) AS venda_diaria
        FROM calendario c
        LEFT JOIN vendas_diarias v ON v.data = c.data
    )
    SELECT
        numero_dia_semana,
        dia_semana,
        COUNT(*) AS dias_no_calendario,
        COUNT(*) FILTER (WHERE venda_diaria = 0) AS dias_sem_venda,
        ROUND(AVG(venda_diaria), 2) AS media_correta,
        ROUND(AVG(venda_diaria) FILTER (WHERE venda_diaria > 0), 2) AS media_so_dias_com_venda
    FROM calendario_com_vendas
    GROUP BY numero_dia_semana, dia_semana
    ORDER BY media_correta ASC, numero_dia_semana ASC
""", pg)

pg.close()
calendario_semana

/var/folders/kc/hb_g153j30gf097ms6nw5w_00000gp/T/ipykernel_7655/385258413.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  calendario_semana = pd.read_sql("""


,numero_dia_semana,dia_semana,dias_no_calendario,dias_sem_venda,media_correta,media_so_dias_com_venda
0,4,Quinta-feira,366,20,157154.32,166238.38
1,7,Domingo,365,12,157616.13,162974.19
2,1,Segunda-feira,365,7,158241.15,161335.26
3,6,Sábado,365,11,164858.27,169980.98
4,2,Terça-feira,365,8,166118.83,169841.38
5,5,Sexta-feira,365,10,170193.68,174987.87
6,3,Quarta-feira,366,10,173605.44,178481.99


**Resultado: pior dia é Quinta-feira**, R$ 157.154,32 — não Domingo, como o estagiário achou. A coluna `media_so_dias_com_venda` (só pra comparação didática, não faz parte da resposta oficial) mostra o efeito do erro: sem os zeros, a Quinta-feira pareceria valer R$ 166.238,38 — 20 das 366 quintas-feiras do período não tiveram nenhuma venda `pos`, e são justamente essas 20 ausências que derrubam a média de verdade. Domingo, aliás, fica em **segundo lugar entre os piores** (R$ 157.616,13) — perto o suficiente da Quinta-feira pra mostrar que a intuição do estagiário não estava tão longe, só olhando a tabela errada.